# Sistema de alerta de riesgo respiratorio desde PPG con machine learning
### Dataset BIDMC (PhysioNet) — se implementa el modelo con mayor AUC en la validación por paciente

**Flujo del notebook**

1. Descarga de las señales PPG y de la referencia clínica de BIDMC (53 pacientes de UCI).
2. Extracción de métricas por ventana de 30 s: HR, HRV (SDNN, RMSSD) y RR. La RR usa relleno con ceros en la FFT para no quedar limitada a múltiplos de 2 rpm.
3. **Etiqueta de riesgo respiratorio** a partir del monitor: RR < 12 o > 20 rpm, o SpO2 < 95 %.
4. Comparación de 7 modelos de ML con **curvas ROC Leave-One-Subject-Out** (cada paciente se evalúa con un modelo que nunca lo vio).
5. **Selección automática del mejor modelo**: 7 modelos × 2 conjuntos de features; gana la combinación con mayor AUC por paciente. Su umbral de decisión se elige con el índice de Youden.
6. `SistemaAlertaRespiratoria`: usa ese modelo ventana a ventana, confirma la alarma con histéresis y **envía la alerta** mediante la función `enviar_alerta()`.
7. Evaluación del sistema completo y demostración en tiempo real sobre la señal cruda.
8. **Interfaz interactiva** (sección 10): monitor en vivo, evaluación manual, análisis de tu propio CSV y configuración.

> El sistema detecta **signos vitales respiratorios fuera de rango** (taquipnea, bradipnea, desaturación), no diagnostica enfermedades respiratorias.

In [ ]:
!pip install requests numpy scipy matplotlib pandas scikit-learn lightgbm xgboost joblib -q

In [ ]:
import os
import json
import warnings
from datetime import datetime
from collections import deque

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.signal import butter, filtfilt, find_peaks

warnings.filterwarnings("ignore")

# ==== Parámetros editables ====
SUJETOS = [f"{i:02d}" for i in range(1, 54)]   # los 53 pacientes de BIDMC
SUJETO_DEMO = "01"                              # paciente para la demostración en tiempo real
VENTANA_S = 30                                  # ventana de análisis (s)
FS_SIG = 125                                    # Hz
CARPETA_DATOS = "bidmc_data"
BASE_URL = "https://physionet.org/files/bidmc/1.0.0/bidmc_csv/"

# Definición de riesgo respiratorio (umbrales NEWS2)
RR_MIN, RR_MAX = 12, 20        # rpm
SPO2_MIN = 95                  # %
INCLUIR_SPO2_EN_ETIQUETA = True   # False = riesgo solo por RR (lo único que el PPG de un canal puede ver)

# Sistema de alerta
N_CONFIRMAR = 2                # ventanas positivas seguidas para activar la alarma (2 x 30 s = 1 min)
N_LIBERAR = 2                  # ventanas negativas seguidas para desactivarla
MAX_RMSSD_RELATIVO = 0.15      # control de calidad de señal
URL_WEBHOOK = None             # ej. "http://192.168.1.50/alerta" para enviar la alerta por HTTP

## 1. Descarga de datos

In [ ]:
def descargar_archivo(nombre, carpeta_datos=CARPETA_DATOS):
    os.makedirs(carpeta_datos, exist_ok=True)
    ruta_local = os.path.join(carpeta_datos, nombre)
    if os.path.exists(ruta_local):
        return ruta_local
    r = requests.get(BASE_URL + nombre, timeout=60)
    r.raise_for_status()
    with open(ruta_local, 'wb') as f:
        f.write(r.content)
    return ruta_local

def leer_csv(nombre):
    descargar_archivo(nombre)
    df = pd.read_csv(os.path.join(CARPETA_DATOS, nombre))
    df.columns = [c.strip() for c in df.columns]
    return df

def cargar_ppg(sid):
    return leer_csv(f"bidmc_{sid}_Signals.csv")['PLETH'].values

def cargar_referencia(sid):
    return leer_csv(f"bidmc_{sid}_Numerics.csv")

## 2. Extracción de métricas por ventana

Igual que en tu notebook, con un cambio en la RR: la FFT de la modulación de amplitud se calcula con **relleno de ceros (×8)**. Con 30 s de señal, la FFT sin relleno solo puede devolver múltiplos de 2 rpm (6, 8, …, 20, 22), así que no distinguía 20 rpm (normal) de 21.4 rpm (riesgo). El relleno no agrega información nueva, pero ubica el pico con una resolución de 0.25 rpm.

In [ ]:
def filtro_pasa_banda(signal, fs, low=0.5, high=8.0, order=3):
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

def normalizar(x):
    return (x - x.mean()) / (x.std() + 1e-9)

RELLENO_FFT = 8

def extraer_metricas_ventana(ppg_win, fs):
    picos, _ = find_peaks(ppg_win, distance=fs*0.4)

    hr, sdnn, rmssd = np.nan, np.nan, np.nan
    if len(picos) > 2:
        intervalos_s = np.diff(picos) / fs
        hr = 60 / intervalos_s.mean()
        intervalos_ms = intervalos_s * 1000
        sdnn = np.std(intervalos_ms, ddof=1)
        rmssd = np.sqrt(np.mean(np.diff(intervalos_ms) ** 2))

    rr = np.nan
    if len(picos) > 3:
        amp_picos = ppg_win[picos]
        t_picos = picos / fs
        t_uniforme = np.arange(0, len(ppg_win) / fs, 1 / fs)
        amp_interp = np.interp(t_uniforme, t_picos, amp_picos)
        amp_interp = amp_interp - amp_interp.mean()
        n_fft = RELLENO_FFT * len(amp_interp)
        freqs = np.fft.rfftfreq(n_fft, d=1/fs)
        fft_vals = np.abs(np.fft.rfft(amp_interp, n=n_fft))
        banda_resp = (freqs >= 0.1) & (freqs <= 0.6)   # 6-36 rpm
        if banda_resp.sum() > 0:
            rr = freqs[banda_resp][np.argmax(fft_vals[banda_resp])] * 60

    return agregar_features({'hr_bpm': hr, 'hrv_sdnn_ms': sdnn, 'hrv_rmssd_ms': rmssd, 'rr_rpm': rr})

def agregar_features(m):
    # rr_fuera_rango: cuántas rpm está la RR FUERA del rango normal [RR_MIN, RR_MAX].
    # Vale 0 dentro del rango y crece hacia ambos lados (RR 8 -> 4, RR 26 -> 6).
    # Le da al modelo la forma de "riesgo de dos lados" que la RR sola no tiene.
    rr = m.get('rr_rpm', np.nan)
    m['rr_fuera_rango'] = np.nan if rr is None or pd.isna(rr) else max(0.0, RR_MIN - rr) + max(0.0, rr - RR_MAX)
    return m

def calidad_ok(m):
    hr, rmssd = m.get('hr_bpm'), m.get('hrv_rmssd_ms')
    if hr is None or pd.isna(hr) or not (30 <= hr <= 200): return False
    if rmssd is None or pd.isna(rmssd): return False
    return rmssd / (60000 / hr) < MAX_RMSSD_RELATIVO

## 3. Dataset multi-paciente con etiqueta de riesgo respiratorio

In [ ]:
def riesgo_respiratorio(rr, spo2):
    riesgo = (rr < RR_MIN) | (rr > RR_MAX)
    if INCLUIR_SPO2_EN_ETIQUETA:
        riesgo = riesgo | (spo2 < SPO2_MIN)
    return riesgo.astype(int)

def procesar_sujeto(sid):
    ppg_norm = normalizar(filtro_pasa_banda(cargar_ppg(sid), FS_SIG))
    num = cargar_referencia(sid)
    tam = VENTANA_S * FS_SIG
    filas = []
    for w in range(len(ppg_norm) // tam):
        fila = extraer_metricas_ventana(ppg_norm[w*tam:(w+1)*tam], FS_SIG)
        t0 = w * VENTANA_S
        m = (num['Time [s]'] >= t0) & (num['Time [s]'] < t0 + VENTANA_S)
        fila.update(sujeto=sid, t_inicio_s=t0,
                    rr_ref=num.loc[m, 'RESP'].mean(),
                    spo2_ref=num.loc[m, 'SpO2'].mean(),
                    hr_ref=num.loc[m, 'HR'].mean())
        filas.append(fila)
    df = pd.DataFrame(filas)
    df['riesgo_resp'] = riesgo_respiratorio(df['rr_ref'], df['spo2_ref'])
    return df

dfs = []
for sid in SUJETOS:
    try:
        d = procesar_sujeto(sid)
        dfs.append(d)
        print(f"[{sid}] ventanas={len(d):2d}  en riesgo={d['riesgo_resp'].sum():2d}  "
              f"RR ref={d['rr_ref'].mean():5.1f}  SpO2 ref min={d['spo2_ref'].min():5.1f}")
    except Exception as e:
        print(f"[{sid}] ERROR: {e}")

df_all = pd.concat(dfs, ignore_index=True)
df_all = df_all.dropna(subset=['rr_ref', 'spo2_ref']).reset_index(drop=True)
df_all.to_csv("dataset_riesgo_respiratorio.csv", index=False)
print(f"\nTotal: {len(df_all)} ventanas de {df_all['sujeto'].nunique()} pacientes")
print(df_all['riesgo_resp'].value_counts().rename({0: 'Normal', 1: 'Riesgo respiratorio'}))

In [ ]:
# ¿Qué tan bien estima el PPG la frecuencia respiratoria?
v = df_all.dropna(subset=['rr_rpm', 'rr_ref'])
mae_rr = (v['rr_rpm'] - v['rr_ref']).abs().mean()
print(f"Error absoluto medio de RR (PPG vs monitor): {mae_rr:.2f} rpm")
print(f"Ventanas con error < 3 rpm: {((v['rr_rpm'] - v['rr_ref']).abs() < 3).mean()*100:.1f} %")

plt.figure(figsize=(6, 6))
plt.scatter(v['rr_ref'], v['rr_rpm'], s=10, alpha=0.5, c=v['riesgo_resp'].map({0: 'tab:green', 1: 'tab:red'}))
plt.plot([5, 35], [5, 35], 'k--', lw=1)
for u in (RR_MIN, RR_MAX):
    plt.axvline(u, color='gray', ls=':'); plt.axhline(u, color='gray', ls=':')
plt.xlabel('RR monitor (rpm)'); plt.ylabel('RR estimada PPG (rpm)')
plt.title('RR estimada vs. referencia (rojo = riesgo respiratorio)')
plt.tight_layout(); plt.savefig('rr_estimada_vs_referencia.png', dpi=130); plt.show()

## 4. Comparación de modelos con curva ROC (validación por paciente)

Los 6 modelos de tu notebook más una regresión logística. Cada paciente se predice con un modelo entrenado **sin** ese paciente, que es lo que ocurriría con un paciente nuevo.

**Por qué 5 grupos estratificados y no Leave-One-Subject-Out.** Al dejar fuera un solo paciente, el conjunto de entrenamiento queda con un poco menos de riesgo justo cuando ese paciente está en riesgo, y un poco más cuando no lo está. Con modelos de poca señal, ese desbalance empuja las probabilidades en la dirección equivocada y hunde el AUC por debajo de 0.5 de forma artificial (es lo que se vio en la corrida anterior, con todos los modelos entre 0.26 y 0.43). `StratifiedGroupKFold` sigue separando por paciente, pero cada grupo conserva la misma proporción de pacientes en riesgo, lo que elimina ese sesgo. El imputador va dentro del pipeline para que tampoco haya fuga de información.

Como referencia se dibuja el punto de la **regla NEWS2 respiratoria** aplicada a la RR estimada (sin aprendizaje).

**Feature nuevo `rr_fuera_rango`.** El riesgo respiratorio es de dos lados: tanto una RR baja (< 12) como una alta (> 20) son peligrosas. Con la RR como número suelto, la mayoría de los modelos no logran aprender esa forma. `rr_fuera_rango` mide cuántas rpm está la RR fuera del rango normal (0 dentro del rango; por ejemplo, 4 si la RR es 8 y 6 si es 26), y así el modelo puede aprenderlo directamente.

Cada modelo se prueba con dos conjuntos de features:

| Conjunto | Features |
|---|---|
| `Todas` | `rr_fuera_rango`, `rr_rpm`, `hr_bpm`, `hrv_sdnn_ms`, `hrv_rmssd_ms` |
| `Solo RR` | `rr_fuera_rango`, `rr_rpm` |

La tabla muestra el recall con umbral 0.5 (columna `Sensibilidad`) y con el umbral de Youden, que es el que usa el sistema de alerta. La gráfica de barras usa el de Youden.

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, f1_score,
                             balanced_accuracy_score)

# Dos conjuntos de features a comparar
FEATURE_SETS = {
    'Todas': ['rr_fuera_rango', 'rr_rpm', 'hr_bpm', 'hrv_sdnn_ms', 'hrv_rmssd_ms'],
    'Solo RR': ['rr_fuera_rango', 'rr_rpm'],
}
if 'rr_fuera_rango' not in df_all.columns:
    df_all = df_all.apply(lambda r: pd.Series(agregar_features(r.to_dict())), axis=1)
y = df_all['riesgo_resp'].values
grupos = df_all['sujeto'].values
pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)

MODELOS = {
    "Regresión logística": Pipeline([("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight='balanced', max_iter=1000))]),
    "SVM": Pipeline([("scaler", StandardScaler()),
        ("clf", SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, class_weight='balanced', random_state=42))]),
    "Random Forest": Pipeline([("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42))]),
    "Decision Tree": Pipeline([("scaler", StandardScaler()),
        ("clf", DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42))]),
    "KNN": Pipeline([("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5, weights='distance'))]),
    "LightGBM": lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                   class_weight='balanced', random_state=42, verbose=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                 scale_pos_weight=pos_weight, random_state=42, eval_metric='logloss'),
}

def con_imputador(modelo):
    return make_pipeline(SimpleImputer(strategy='median').set_output(transform="pandas"), clone(modelo))

def metricas_alarma(y_true, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'Sensibilidad': tp / max(tp + fn, 1), 'Especificidad': tn / max(tn + fp, 1),
            'Precisión': tp / max(tp + fp, 1), 'F1': f1_score(y_true, pred, zero_division=0),
            'Bal. accuracy': balanced_accuracy_score(y_true, pred),
            'Falsas alarmas/h': fp / (len(y_true) * VENTANA_S / 3600)}

def umbral_youden(y_true, proba):
    fpr, tpr, umbrales = roc_curve(y_true, proba)
    i = np.argmax(tpr - fpr)
    return float(np.clip(umbrales[i], 0, 1))

# Validación por paciente en 5 grupos estratificados: cada paciente cae en un solo grupo (nunca está
# a la vez en entrenamiento y prueba) y cada grupo conserva la proporción de pacientes en riesgo.
cv_pacientes = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
proba_loso, filas = {}, []
for nombre_set, cols in FEATURE_SETS.items():
    X_set = df_all[cols].astype(float).replace([np.inf, -np.inf], np.nan)
    for nombre, modelo in MODELOS.items():
        clave = f"{nombre} | {nombre_set}"
        p = cross_val_predict(con_imputador(modelo), X_set, y, cv=cv_pacientes, groups=grupos, method='predict_proba')[:, 1]
        proba_loso[clave] = p
        fila = {'Candidato': clave, 'Modelo': nombre, 'Features': nombre_set,
                'AUC (por paciente)': roc_auc_score(y, p),
                'Recall (umbral Youden)': metricas_alarma(y, (p >= umbral_youden(y, p)).astype(int))['Sensibilidad']}
        fila.update(metricas_alarma(y, (p >= 0.5).astype(int)))
        filas.append(fila)
        print(f"{clave:32s} AUC = {fila['AUC (por paciente)']:.3f}")

# Línea base sin aprendizaje: regla NEWS2 sobre la RR estimada
regla = ((df_all['rr_rpm'] < RR_MIN) | (df_all['rr_rpm'] > RR_MAX)).astype(int).values
fila = {'Candidato': 'Regla NEWS2 (RR estimada)', 'Modelo': 'Regla NEWS2', 'Features': '—',
        'AUC (por paciente)': np.nan}
fila.update(metricas_alarma(y, regla))
fila['Recall (umbral Youden)'] = fila['Sensibilidad']
filas.append(fila)

df_roc = pd.DataFrame(filas).set_index('Candidato').sort_values('AUC (por paciente)', ascending=False)
print("\n=== COMPARATIVO (Sensibilidad/Especificidad/... con umbral 0.5) ===")
print(df_roc.drop(columns=['Modelo', 'Features']).round(3).to_string())
df_roc.to_csv('comparativo_roc_respiratorio.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Azar (AUC = 0.50)')
# Para cada modelo se dibuja su mejor conjunto de features
mejor_por_modelo = (df_roc.dropna(subset=['AUC (por paciente)'])
                    .sort_values('AUC (por paciente)', ascending=False)
                    .drop_duplicates('Modelo'))
for clave in mejor_por_modelo.index:
    fpr, tpr, _ = roc_curve(y, proba_loso[clave])
    ax.plot(fpr, tpr, lw=2, label=f"{clave} (AUC = {roc_auc_score(y, proba_loso[clave]):.3f})")
r = df_roc.loc['Regla NEWS2 (RR estimada)']
ax.scatter([1 - r['Especificidad']], [r['Sensibilidad']], marker='*', s=200, color='black', zorder=5,
           label='Regla NEWS2 (RR estimada)')
ax.set_xlabel('Tasa de falsos positivos (FPR)'); ax.set_ylabel('Tasa de verdaderos positivos (TPR)')
ax.set_title('ROC — riesgo respiratorio, validación por paciente')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('roc_riesgo_respiratorio.png', dpi=130); plt.show()

In [ ]:
# Comparativo AUC vs. Recall por modelo
comp = mejor_por_modelo[['AUC (por paciente)', 'Recall (umbral Youden)']]
ganador = comp['AUC (por paciente)'].idxmax()
x = np.arange(len(comp)); ancho = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
colores_auc = ['tab:red' if n == ganador else 'tab:blue' for n in comp.index]
b1 = ax.bar(x - ancho/2, comp['AUC (por paciente)'], ancho, label='AUC', color=colores_auc)
b2 = ax.bar(x + ancho/2, comp['Recall (umbral Youden)'], ancho, label='Recall (umbral Youden)', color='tab:orange')
ax.bar_label(b1, fmt='%.3f', fontsize=8); ax.bar_label(b2, fmt='%.3f', fontsize=8)
ax.axhline(0.5, color='k', ls='--', lw=1, label='Azar (AUC 0.5)')
ax.axhline(df_roc.loc['Regla NEWS2 (RR estimada)', 'Sensibilidad'], color='tab:green',
           ls=':', lw=2, label='Recall regla NEWS2')
ax.set_xticks(x); ax.set_xticklabels([c.replace(' | ', '\n') for c in comp.index], fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel('Valor')
ax.set_title(f'AUC y Recall por modelo (validación por paciente) — mejor en rojo: {ganador}')
ax.legend(); plt.tight_layout(); plt.savefig('comparativo_auc_recall.png', dpi=130); plt.show()

## 5. Selección del mejor modelo y de su umbral

- **Modelo:** la combinación modelo + conjunto de features con **mayor AUC por paciente**. Se elige automáticamente: si vuelves a correr con otros datos o parámetros, el sistema se actualiza solo.
- **Umbral:** el que maximiza el índice de Youden (sensibilidad + especificidad − 1). El umbral de 0.5 no tiene por qué ser el mejor para una alarma.

Para medir el rendimiento sin trampa, el umbral se elige de forma **anidada**: en cada uno de los 5 grupos de pacientes, con validación cruzada interna solo sobre los pacientes de entrenamiento.

In [ ]:
MEJOR_CLAVE = df_roc['AUC (por paciente)'].idxmax()
MEJOR_MODELO = df_roc.loc[MEJOR_CLAVE, 'Modelo']
SET_FEATURES = df_roc.loc[MEJOR_CLAVE, 'Features']
FEATURES = FEATURE_SETS[SET_FEATURES]
X = df_all[FEATURES].astype(float).replace([np.inf, -np.inf], np.nan)
AUC_MEJOR = df_roc.loc[MEJOR_CLAVE, 'AUC (por paciente)']
print(f"Mejor modelo según la curva ROC: {MEJOR_MODELO} con features '{SET_FEATURES}' {FEATURES}")
print(f"AUC por paciente = {AUC_MEJOR:.3f}")
if AUC_MEJOR < 0.6:
    print("ATENCIÓN: AUC < 0.6. El modelo apenas distingue pacientes en riesgo de pacientes normales; "
          "revisa la comparación con la regla NEWS2 antes de usarlo como única decisión.")

# Evaluación anidada: por cada paciente, modelo y umbral obtenidos SIN ese paciente
modelos_por_paciente, umbral_por_paciente = {}, {}
proba_anidada = np.zeros(len(y))
# Mismos 5 grupos de pacientes que en la sección 4; el umbral se elige con validación interna
# solo sobre los pacientes de entrenamiento de cada grupo.
for idx_tr, idx_te in cv_pacientes.split(X, y, groups=grupos):
    Xtr, ytr, gtr = X.iloc[idx_tr], y[idx_tr], grupos[idx_tr]
    n_folds = min(5, len(np.unique(gtr)))
    p_int = cross_val_predict(con_imputador(MODELOS[MEJOR_MODELO]), Xtr, ytr,
                              cv=GroupKFold(n_splits=n_folds), groups=gtr, method='predict_proba')[:, 1]
    umbral = umbral_youden(ytr, p_int)
    modelo = con_imputador(MODELOS[MEJOR_MODELO]).fit(Xtr, ytr)
    proba_anidada[idx_te] = modelo.predict_proba(X.iloc[idx_te])[:, 1]
    for sid in np.unique(grupos[idx_te]):
        modelos_por_paciente[sid], umbral_por_paciente[sid] = modelo, umbral

pred_anidada = np.array([proba_anidada[i] >= umbral_por_paciente[g] for i, g in enumerate(grupos)]).astype(int)
print("\nRendimiento por ventana con umbral anidado:")
for k, v in metricas_alarma(y, pred_anidada).items():
    print(f"  {k:18s} {v:.3f}")

# Modelo y umbral finales, entrenados con TODOS los pacientes
UMBRAL_FINAL = umbral_youden(y, proba_loso[MEJOR_CLAVE])
MODELO_FINAL = con_imputador(MODELOS[MEJOR_MODELO]).fit(X, y)
print(f"\nUmbral final (Youden sobre la ROC por paciente): {UMBRAL_FINAL:.3f}")

# ¿Supera a la regla NEWS2?
ba_modelo = balanced_accuracy_score(y, pred_anidada)
ba_regla = balanced_accuracy_score(y, regla)
print(f"\nAccuracy balanceada — {MEJOR_MODELO}: {ba_modelo:.3f} | regla NEWS2: {ba_regla:.3f}")
if ba_modelo < ba_regla:
    print("El modelo NO supera a la regla NEWS2 en esta corrida: repórtalo así en el informe.")

## 6. Sistema de alerta respiratoria

`SistemaAlertaRespiratoria` procesa una ventana de 30 s cada vez:

1. Extrae HR, HRV y RR del PPG y revisa la **calidad de la señal**. Una ventana con mala señal no se usa para decidir y el estado anterior se mantiene.
2. El **modelo de ML** calcula la probabilidad de riesgo respiratorio y la compara con el umbral.
3. **Histéresis:** la alarma se activa tras `N_CONFIRMAR` ventanas positivas seguidas y se desactiva tras `N_LIBERAR` negativas. Evita alarmas intermitentes por ruido.
4. **Banderas rojas:** RR ≤ 8 o ≥ 25 rpm (3 puntos en NEWS2), o SpO2 ≤ 91 % si hay sensor, activan la alarma **de inmediato**, sin esperar confirmación. Es una red de seguridad para valores extremos, que son justo los que no se pueden perder.
5. Cada vez que la alarma se **activa**, **escala a crítico** o se **desactiva**, se llama a `enviar_alerta()`. Por defecto imprime un mensaje; si defines `URL_WEBHOOK`, además hace un POST HTTP con la alerta en JSON (a un ESP32, un servidor, un bot, etc.).

Niveles: `Normal` → `Vigilancia` (probabilidad sobre el umbral, falta confirmar) → `ALERTA` (confirmada) → `CRÍTICO` (bandera roja).

In [ ]:
def enviar_alerta(evento):
    icono = {'ALERTA': '🟠', 'CRÍTICO': '🔴', 'FIN DE ALERTA': '🟢'}.get(evento['tipo'], '•')
    print(f"\n{icono} [{evento['hora']}] {evento['tipo']} — paciente {evento['paciente']} (t = {evento['t_s']} s)")
    print(f"   RR = {evento['rr_rpm']} rpm | HR = {evento['hr_bpm']} lpm | SpO2 = {evento['spo2']} | "
          f"prob. riesgo = {evento['probabilidad']}")
    print(f"   Motivo: {evento['motivo']}\n")
    if URL_WEBHOOK:
        try:
            requests.post(URL_WEBHOOK, json=evento, timeout=3)
        except requests.RequestException as e:
            print(f"   (no se pudo enviar al webhook: {e})")


class SistemaAlertaRespiratoria:
    def __init__(self, modelo, umbral, paciente='?', n_confirmar=N_CONFIRMAR, n_liberar=N_LIBERAR,
                 notificar=enviar_alerta):
        self.modelo, self.umbral, self.paciente = modelo, umbral, paciente
        self.n_confirmar, self.n_liberar, self.notificar = n_confirmar, n_liberar, notificar
        self.reset()

    def reset(self):
        self.alarma, self.critico = False, False
        self._pos, self._neg = 0, 0
        self.historial = []

    @staticmethod
    def _banderas_rojas(rr, spo2):
        motivos = []
        if not pd.isna(rr) and rr <= 8:  motivos.append(f"bradipnea severa (RR {rr:.1f} ≤ 8)")
        if not pd.isna(rr) and rr >= 25: motivos.append(f"taquipnea severa (RR {rr:.1f} ≥ 25)")
        if spo2 is not None and not pd.isna(spo2) and spo2 <= 91:
            motivos.append(f"desaturación severa (SpO2 {spo2:.1f} ≤ 91)")
        return motivos

    @staticmethod
    def _motivo_modelo(rr, spo2):
        partes = []
        if not pd.isna(rr) and rr > RR_MAX: partes.append(f"RR {rr:.1f} > {RR_MAX}")
        if not pd.isna(rr) and rr < RR_MIN: partes.append(f"RR {rr:.1f} < {RR_MIN}")
        if spo2 is not None and not pd.isna(spo2) and spo2 < SPO2_MIN: partes.append(f"SpO2 {spo2:.1f} < {SPO2_MIN}")
        return "patrón de riesgo detectado por el modelo" + (f" ({', '.join(partes)})" if partes else "")

    def _notificar(self, tipo, motivo, estado):
        self.notificar({
            'tipo': tipo, 'paciente': self.paciente, 't_s': estado['t_s'],
            'hora': datetime.now().strftime('%H:%M:%S'), 'motivo': motivo,
            'rr_rpm': None if pd.isna(estado['rr_rpm']) else round(estado['rr_rpm'], 1),
            'hr_bpm': None if pd.isna(estado['hr_bpm']) else round(estado['hr_bpm'], 1),
            'spo2': estado['spo2'],
            'probabilidad': None if pd.isna(estado['probabilidad']) else round(estado['probabilidad'], 3),
        })

    def actualizar_metricas(self, metricas, t=None, spo2=None):
        metricas = agregar_features(dict(metricas))
        rr, hr = metricas.get('rr_rpm', np.nan), metricas.get('hr_bpm', np.nan)
        ok = calidad_ok(metricas)
        prob, positivo = np.nan, None
        banderas = self._banderas_rojas(rr, spo2) if ok else []

        if ok:
            fila = pd.DataFrame([{f: metricas.get(f, np.nan) for f in FEATURES}])
            prob = float(self.modelo.predict_proba(fila)[0, 1])
            positivo = prob >= self.umbral or bool(banderas) or \
                       (spo2 is not None and not pd.isna(spo2) and spo2 < SPO2_MIN)
            if positivo:
                self._pos, self._neg = self._pos + 1, 0
            else:
                self._neg, self._pos = self._neg + 1, 0

        estado = {'t_s': t, 'rr_rpm': rr, 'hr_bpm': hr, 'spo2': spo2, 'probabilidad': prob,
                  'calidad': 'OK' if ok else 'Señal pobre', 'positivo': positivo}

        # Activación / escalamiento / desactivación
        if banderas and not self.critico:
            self.alarma, self.critico = True, True
            self._notificar('CRÍTICO', '; '.join(banderas), estado)
        elif not self.alarma and self._pos >= self.n_confirmar:
            self.alarma = True
            self._notificar('ALERTA', self._motivo_modelo(rr, spo2), estado)
        elif self.alarma and self._neg >= self.n_liberar:
            self.alarma, self.critico = False, False
            self._notificar('FIN DE ALERTA', 'parámetros respiratorios de vuelta en rango', estado)
        elif self.critico and ok and not banderas:
            self.critico = False   # sigue en alerta, pero ya sin bandera roja

        if self.critico:        nivel = 'CRÍTICO'
        elif self.alarma:       nivel = 'ALERTA'
        elif positivo:          nivel = 'Vigilancia'
        else:                   nivel = 'Normal'
        estado.update(alarma=int(self.alarma), nivel=nivel)
        self.historial.append(estado)
        return estado

    def actualizar(self, ppg_ventana_cruda, fs, t=None, spo2=None):
        x = normalizar(filtro_pasa_banda(np.asarray(ppg_ventana_cruda, dtype=float), fs))
        return self.actualizar_metricas(extraer_metricas_ventana(x, fs), t=t, spo2=spo2)

    def tabla(self):
        return pd.DataFrame(self.historial)

## 7. Evaluación del sistema completo (paciente por paciente)

Cada paciente se procesa como un flujo de ventanas, con el modelo y el umbral obtenidos **sin** él (sección 5). Se compara con la decisión del modelo ventana a ventana, sin histéresis ni banderas rojas.

In [ ]:
silencio = lambda evento: None   # durante la evaluación no se imprimen las alertas

registros = []
for sid, g in df_all.groupby('sujeto', sort=False):
    s = SistemaAlertaRespiratoria(modelos_por_paciente[sid], umbral_por_paciente[sid], paciente=sid, notificar=silencio)
    for _, fila in g.sort_values('t_inicio_s').iterrows():
        e = s.actualizar_metricas(fila.to_dict(), t=fila['t_inicio_s'])
        e.update(sujeto=sid, riesgo_resp=fila['riesgo_resp'])
        registros.append(e)
eval_sis = pd.DataFrame(registros)

df_final = pd.DataFrame({
    f'{MEJOR_MODELO} por ventana (umbral anidado)': metricas_alarma(y, pred_anidada),
    f'Sistema de alerta ({MEJOR_MODELO} + histéresis + banderas rojas)':
        metricas_alarma(eval_sis['riesgo_resp'].values, eval_sis['alarma'].values),
    'Regla NEWS2 (RR estimada)': metricas_alarma(y, regla),
}).T
print(df_final.round(3).to_string())
print(f"\nVentanas descartadas por mala señal: {(eval_sis['calidad'] != 'OK').mean()*100:.1f} %")
df_final.to_csv('evaluacion_sistema_alerta_respiratoria.csv')

# Detección por paciente: ¿el sistema alertó en algún momento a los pacientes en riesgo?
por_paciente = eval_sis.groupby('sujeto').agg(en_riesgo=('riesgo_resp', 'max'), alerto=('alarma', 'max'))
tp = ((por_paciente.en_riesgo == 1) & (por_paciente.alerto == 1)).sum()
fn = ((por_paciente.en_riesgo == 1) & (por_paciente.alerto == 0)).sum()
fp = ((por_paciente.en_riesgo == 0) & (por_paciente.alerto == 1)).sum()
tn = ((por_paciente.en_riesgo == 0) & (por_paciente.alerto == 0)).sum()
print(f"\nPor paciente: detectados {tp} de {tp+fn} en riesgo | alarmas falsas en {fp} de {fp+tn} normales")

## 8. Guardar el sistema

In [ ]:
joblib.dump(MODELO_FINAL, 'modelo_alerta_respiratoria.joblib')
config = {'modelo': MEJOR_MODELO, 'conjunto_features': SET_FEATURES, 'auc_por_paciente': round(float(AUC_MEJOR), 4), 'umbral': UMBRAL_FINAL,
          'features': FEATURES, 'ventana_s': VENTANA_S, 'fs': FS_SIG, 'rr_min': RR_MIN, 'rr_max': RR_MAX,
          'spo2_min': SPO2_MIN, 'n_confirmar': N_CONFIRMAR, 'n_liberar': N_LIBERAR,
          'max_rmssd_relativo': MAX_RMSSD_RELATIVO, 'relleno_fft': RELLENO_FFT}
with open('config_alerta_respiratoria.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Guardados: modelo_alerta_respiratoria.joblib y config_alerta_respiratoria.json")
print(config)

# Para usarlo en otro script:
#   modelo = joblib.load('modelo_alerta_respiratoria.joblib')
#   cfg = json.load(open('config_alerta_respiratoria.json'))
#   sistema = SistemaAlertaRespiratoria(modelo, cfg['umbral'], paciente='cama 3')
#   estado = sistema.actualizar(ppg_30s, fs=125)

## 9. Demostración en tiempo real

Simula un monitor: cada 30 s entra un bloque de PPG crudo del paciente `SUJETO_DEMO`, el sistema decide y **envía la alerta** cuando corresponde. Si el paciente no dispara ninguna alerta, prueba con uno que el dataset marque en riesgo (columna `en riesgo` de la sección 3).

In [ ]:
sistema = SistemaAlertaRespiratoria(MODELO_FINAL, UMBRAL_FINAL, paciente=SUJETO_DEMO)
ppg_stream = cargar_ppg(SUJETO_DEMO)
tam = VENTANA_S * FS_SIG

print(f"Monitoreando paciente {SUJETO_DEMO} con {MEJOR_MODELO} (umbral {UMBRAL_FINAL:.3f})\n")
for w in range(len(ppg_stream) // tam):
    e = sistema.actualizar(ppg_stream[w*tam:(w+1)*tam], FS_SIG, t=w * VENTANA_S)
    prob = '  -  ' if pd.isna(e['probabilidad']) else f"{e['probabilidad']:.3f}"
    print(f"t={e['t_s']:4d}s  RR={e['rr_rpm']:5.1f}  HR={e['hr_bpm']:6.1f}  "
          f"prob={prob}  {e['calidad']:11s}  {e['nivel']}")

hist = sistema.tabla()
colores = {'Normal': 'tab:green', 'Vigilancia': 'gold', 'ALERTA': 'tab:orange', 'CRÍTICO': 'tab:red'}
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(hist['t_s'], hist['rr_rpm'], 'k-o', ms=4)
axes[0].axhspan(RR_MIN, RR_MAX, color='green', alpha=0.1); axes[0].set_ylabel('RR (rpm)')
axes[1].plot(hist['t_s'], hist['probabilidad'], 'b-o', ms=4)
axes[1].axhline(UMBRAL_FINAL, color='red', ls='--', label='Umbral'); axes[1].legend()
axes[1].set_ylabel('Prob. riesgo'); axes[1].set_ylim(0, 1)
axes[2].bar(hist['t_s'], 1, width=VENTANA_S * 0.95, align='edge', color=hist['nivel'].map(colores))
for n, c in colores.items():
    axes[2].bar(0, 0, color=c, label=n)
axes[2].legend(ncol=4, fontsize=8, loc='upper right'); axes[2].set_yticks([]); axes[2].set_xlabel('Tiempo (s)')
axes[0].set_title(f'Alerta respiratoria en tiempo real — paciente {SUJETO_DEMO} ({MEJOR_MODELO})')
plt.tight_layout(); plt.savefig(f'alerta_tiempo_real_{SUJETO_DEMO}.png', dpi=130); plt.show()

## 10. Interfaz interactiva

Panel con 4 pestañas para usar el sistema de alerta sin tocar código. Funciona en Google Colab y Jupyter (usa `ipywidgets`).

| Pestaña | Qué hace |
|---|---|
| **Monitor en vivo** | Eliges un paciente de BIDMC y el sistema lo monitorea ventana a ventana, con semáforo de nivel, gráfica que se actualiza y registro de alertas. Puedes simular un sensor de SpO2 con los valores del monitor. |
| **Evaluación manual** | Mueves los deslizadores de RR, HR, SDNN, RMSSD y SpO2, y ves al instante la probabilidad del modelo, la decisión y los puntos NEWS2. Sirve para explorar cómo responde el modelo. |
| **Analizar mi señal** | Subes un CSV con tu propio PPG (por ejemplo, de un MAX30102) y el sistema lo analiza completo. |
| **Configuración** | Umbral del modelo, ventanas para confirmar y liberar la alarma, y URL del webhook para enviar las alertas. |

**Arranque rápido.** Entrenar los modelos (secciones 3 a 5) tarda varios minutos, sobre todo la primera vez por la descarga de los 53 pacientes. Una vez que la sección 8 guardó `modelo_alerta_respiratoria.joblib` y `config_alerta_respiratoria.json`, en sesiones siguientes basta con correr: **configuración → secciones 1, 2 y 6 → sección 10**. La primera celda de esta sección carga el modelo desde disco.

**En Google Colab** el monitor corre en el hilo principal (`USAR_HILO = False` automático), porque desde un hilo en segundo plano los widgets se quedan en "Cargando señal…". La consecuencia es que el botón **Detener** no puede interrumpir un monitoreo en curso; para verlo completo rápido, pon la velocidad en 0 s por ventana.

**No instales ni actualices `ipywidgets` en Colab**: ya viene instalado, y una versión distinta hace que los widgets se queden en "Loading widget…" para siempre.

In [ ]:
# ============================================================
# 10. INTERFAZ INTERACTIVA — carga rápida del modelo guardado
# ============================================================
# Si ya entrenaste antes (sección 8 creó los archivos), no hace falta repetir las secciones 3 a 7:
# esta celda carga el modelo y el umbral desde disco.
import os, json, joblib
if 'MODELO_FINAL' not in globals():
    if os.path.exists('modelo_alerta_respiratoria.joblib') and os.path.exists('config_alerta_respiratoria.json'):
        MODELO_FINAL = joblib.load('modelo_alerta_respiratoria.joblib')
        _cfg = json.load(open('config_alerta_respiratoria.json', encoding='utf-8'))
        MEJOR_MODELO, AUC_MEJOR, UMBRAL_FINAL = _cfg['modelo'], _cfg['auc_por_paciente'], _cfg['umbral']
        FEATURES = _cfg['features']
        print(f"Modelo cargado desde disco: {MEJOR_MODELO} (AUC {AUC_MEJOR:.3f}, umbral {UMBRAL_FINAL:.3f})")
    else:
        raise RuntimeError("No hay modelo entrenado: corre primero las secciones 3 a 5 y la 8 (guardado).")
else:
    print(f"Usando el modelo ya entrenado en esta sesión: {MEJOR_MODELO} (umbral {UMBRAL_FINAL:.3f})")

In [ ]:
import io
import time
import threading
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure

import sys
EN_COLAB = 'google.colab' in sys.modules
# En Colab los widgets no se refrescan de forma fiable desde un hilo en segundo plano:
# se queda pegado en "Cargando señal…". Por eso ahí el monitor corre en el hilo principal.
USAR_HILO = not EN_COLAB

COLORES_NIVEL = {'Normal': '#2e7d32', 'Vigilancia': '#f9a825', 'ALERTA': '#ef6c00',
                 'CRÍTICO': '#c62828', 'Sin datos': '#757575'}
ICONOS = {'ALERTA': '🟠', 'CRÍTICO': '🔴', 'FIN DE ALERTA': '🟢'}

# ---------- Lógica (independiente de los widgets) ----------
def puntos_rr_news2(rr):
    if rr is None or pd.isna(rr): return None
    if rr <= 8: return 3
    if rr <= 11: return 1
    if rr <= 20: return 0
    if rr <= 24: return 2
    return 3

def puntos_spo2_news2(s):
    if s is None or pd.isna(s): return None
    if s <= 91: return 3
    if s <= 93: return 2
    if s <= 95: return 1
    return 0

def evaluar_manual(rr, hr, sdnn, rmssd, spo2=None, umbral=None, modelo=None):
    umbral = UMBRAL_FINAL if umbral is None else umbral
    modelo = MODELO_FINAL if modelo is None else modelo
    m = agregar_features({'rr_rpm': rr, 'hr_bpm': hr, 'hrv_sdnn_ms': sdnn, 'hrv_rmssd_ms': rmssd})
    prob = float(modelo.predict_proba(pd.DataFrame([{f: m[f] for f in FEATURES}]))[0, 1])
    banderas = SistemaAlertaRespiratoria._banderas_rojas(rr, spo2)
    spo2_baja = spo2 is not None and spo2 < SPO2_MIN
    positivo = prob >= umbral or bool(banderas) or spo2_baja
    nivel = 'CRÍTICO' if banderas else ('ALERTA' if positivo else 'Normal')
    return {'probabilidad': prob, 'umbral': umbral, 'positivo': positivo, 'nivel': nivel,
            'banderas': banderas, 'calidad_ok': calidad_ok(m),
            'puntos_rr': puntos_rr_news2(rr), 'puntos_spo2': puntos_spo2_news2(spo2)}

def ejecutar_sobre_senal(ppg, fs, paciente, spo2_por_ventana=None, umbral=None, n_confirmar=None,
                         n_liberar=None, notificar=None, al_actualizar=None, pausa=0.0, detener=lambda: False):
    sistema = SistemaAlertaRespiratoria(
        MODELO_FINAL, UMBRAL_FINAL if umbral is None else umbral, paciente=paciente,
        n_confirmar=N_CONFIRMAR if n_confirmar is None else n_confirmar,
        n_liberar=N_LIBERAR if n_liberar is None else n_liberar,
        notificar=notificar or (lambda e: None))
    tam = int(VENTANA_S * fs)
    for w in range(len(ppg) // tam):
        if detener():
            break
        spo2 = None
        if spo2_por_ventana is not None and w < len(spo2_por_ventana) and not pd.isna(spo2_por_ventana[w]):
            spo2 = float(spo2_por_ventana[w])
        estado = sistema.actualizar(ppg[w*tam:(w+1)*tam], fs, t=w * VENTANA_S, spo2=spo2)
        if al_actualizar:
            al_actualizar(sistema, estado)
        if pausa > 0:
            time.sleep(pausa)
    return sistema

def spo2_monitor_por_ventana(sid):
    num = cargar_referencia(sid)
    n = int(num['Time [s]'].max() // VENTANA_S) + 1
    return [num.loc[(num['Time [s]'] >= w*VENTANA_S) & (num['Time [s]'] < (w+1)*VENTANA_S), 'SpO2'].mean()
            for w in range(n)]

def leer_archivo_subido(valor):
    # Compatible con ipywidgets 7 (dict) y 8 (tupla)
    if isinstance(valor, dict):
        item = list(valor.values())[0]
        return item['metadata']['name'], bytes(item['content'])
    item = valor[0]
    return item['name'], bytes(item['content'])

# ---------- Presentación ----------
def fmt(v, dec=1):
    return '—' if v is None or pd.isna(v) else f"{v:.{dec}f}"

def html_estado(estado, paciente, umbral):
    nivel = estado.get('nivel', 'Sin datos') if estado else 'Sin datos'
    color = COLORES_NIVEL.get(nivel, '#757575')
    if not estado:
        detalle = 'Esperando datos…'
    else:
        detalle = (f"t = {estado['t_s']} s &nbsp;|&nbsp; RR <b>{fmt(estado['rr_rpm'])}</b> rpm &nbsp;|&nbsp; "
                   f"HR <b>{fmt(estado['hr_bpm'])}</b> lpm &nbsp;|&nbsp; SpO2 <b>{fmt(estado['spo2'])}</b> % "
                   f"&nbsp;|&nbsp; P(riesgo) {MEJOR_MODELO} <b>{fmt(estado['probabilidad'], 3)}</b> (umbral {umbral:.3f})"
                   f"&nbsp;|&nbsp; señal: {estado['calidad']}")
    return (f"<div style='border-radius:10px;padding:14px 18px;background:{color};color:white;font-family:sans-serif'>"
            f"<div style='font-size:13px;opacity:.9'>Paciente {paciente}</div>"
            f"<div style='font-size:30px;font-weight:bold;letter-spacing:1px'>{nivel}</div>"
            f"<div style='font-size:13px;margin-top:4px'>{detalle}</div></div>")

def html_log(eventos):
    if not eventos:
        return "<div style='font-family:sans-serif;color:#666'>Sin alertas.</div>"
    filas = "".join(
        f"<div style='padding:6px 8px;border-bottom:1px solid #ddd'>{ICONOS.get(e['tipo'], '•')} "
        f"<b>{e['tipo']}</b> — t = {e['t_s']} s ({e['hora']})<br>"
        f"<span style='font-size:12px;color:#555'>{e['motivo']} · RR {e['rr_rpm']} · HR {e['hr_bpm']} · "
        f"SpO2 {e['spo2']} · P {e['probabilidad']}</span></div>"
        for e in reversed(eventos))
    return (f"<div style='font-family:sans-serif;max-height:260px;overflow-y:auto;border:1px solid #ccc;"
            f"border-radius:8px'>{filas}</div>")

def figura_png(hist, umbral, titulo):
    fig = Figure(figsize=(9, 5.2))
    ax = fig.subplots(3, 1, sharex=True)
    if len(hist):
        t = hist['t_s']
        ax[0].plot(t, hist['rr_rpm'], 'k-o', ms=4)
        ax[1].plot(t, hist['probabilidad'], 'b-o', ms=4)
        ax[2].bar(t, 1, width=VENTANA_S * 0.95, align='edge', color=hist['nivel'].map(COLORES_NIVEL))
    ax[0].axhspan(RR_MIN, RR_MAX, color='green', alpha=0.1); ax[0].set_ylabel('RR (rpm)')
    ax[1].axhline(umbral, color='red', ls='--', lw=1); ax[1].set_ylim(0, 1); ax[1].set_ylabel('P(riesgo)')
    ax[2].set_yticks([]); ax[2].set_ylabel('Nivel'); ax[2].set_xlabel('Tiempo (s)')
    ax[0].set_title(titulo, fontsize=10)
    fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=90)
    return buf.getvalue()

def enviar_webhook(evento, url):
    if url:
        try:
            requests.post(url, json=evento, timeout=3)
        except requests.RequestException:
            pass

In [ ]:
# ---------- Pestaña: Configuración ----------
sl_umbral = widgets.FloatSlider(value=round(UMBRAL_FINAL, 3), min=0.0, max=1.0, step=0.005,
                                description='Umbral modelo', readout_format='.3f',
                                style={'description_width': '130px'}, layout=widgets.Layout(width='480px'))
sl_confirmar = widgets.IntSlider(value=N_CONFIRMAR, min=1, max=5, description='Ventanas p/ activar',
                                 style={'description_width': '130px'})
sl_liberar = widgets.IntSlider(value=N_LIBERAR, min=1, max=5, description='Ventanas p/ liberar',
                               style={'description_width': '130px'})
txt_webhook = widgets.Text(value=URL_WEBHOOK or '', placeholder='http://192.168.1.50/alerta',
                           description='Webhook (URL)', style={'description_width': '130px'},
                           layout=widgets.Layout(width='480px'))
btn_youden = widgets.Button(description='Restablecer umbral de Youden', icon='undo')
btn_youden.on_click(lambda _: setattr(sl_umbral, 'value', round(UMBRAL_FINAL, 3)))
tab_config = widgets.VBox([
    widgets.HTML("<b>Parámetros del sistema de alerta</b><br><span style='color:#555'>Subir el umbral reduce falsas "
                 "alarmas pero pierde detecciones; bajarlo hace lo contrario.</span>"),
    sl_umbral, btn_youden, sl_confirmar, sl_liberar, txt_webhook])

def parametros_actuales():
    return dict(umbral=sl_umbral.value, n_confirmar=sl_confirmar.value, n_liberar=sl_liberar.value)

# ---------- Pestaña: Monitor en vivo ----------
dd_paciente = widgets.Dropdown(options=SUJETOS, value=SUJETO_DEMO, description='Paciente')
sl_velocidad = widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1, description='s por ventana',
                                   style={'description_width': '100px'})
chk_spo2 = widgets.Checkbox(value=False, description='Simular sensor de SpO2 (usar SpO2 del monitor)',
                            layout=widgets.Layout(width='420px'))
btn_iniciar = widgets.Button(description='Iniciar', button_style='success', icon='play')
btn_detener = widgets.Button(description='Detener', button_style='danger', icon='stop', disabled=True)
html_mon_estado = widgets.HTML(html_estado(None, SUJETO_DEMO, UMBRAL_FINAL))
img_mon = widgets.Image(format='png', layout=widgets.Layout(width='100%', max_width='820px'))
html_mon_log = widgets.HTML(html_log([]))
control_mon = {'detener': False, 'hilo': None}

def correr_monitor():
    pid, p = dd_paciente.value, parametros_actuales()
    eventos = []
    def notificar(e):
        eventos.append(e)
        html_mon_log.value = html_log(eventos)
        enviar_webhook(e, txt_webhook.value.strip())
    def al_actualizar(sistema, estado):
        html_mon_estado.value = html_estado(estado, pid, p['umbral'])
        img_mon.value = figura_png(sistema.tabla(), p['umbral'], f'Paciente {pid} — {MEJOR_MODELO}')
    try:
        html_mon_estado.value = html_estado(None, pid, p['umbral']).replace('Esperando datos…', 'Cargando señal…')
        ppg = cargar_ppg(pid)
        spo2 = spo2_monitor_por_ventana(pid) if chk_spo2.value else None
        ejecutar_sobre_senal(ppg, FS_SIG, pid, spo2_por_ventana=spo2, notificar=notificar,
                             al_actualizar=al_actualizar, pausa=sl_velocidad.value,
                             detener=lambda: control_mon['detener'], **p)
        if not eventos:
            html_mon_log.value = html_log([]).replace('Sin alertas.', 'Monitoreo terminado: sin alertas.')
    except Exception as ex:
        html_mon_log.value = f"<div style='color:#c62828'>Error: {ex}</div>"
    finally:
        btn_iniciar.disabled, btn_detener.disabled = False, True

def al_iniciar(_):
    control_mon['detener'] = False
    btn_iniciar.disabled, btn_detener.disabled = True, False
    html_mon_log.value = html_log([])
    if USAR_HILO:
        control_mon['hilo'] = threading.Thread(target=correr_monitor, daemon=True)
        control_mon['hilo'].start()
    else:
        correr_monitor()

def al_detener(_):
    control_mon['detener'] = True

btn_iniciar.on_click(al_iniciar)
btn_detener.on_click(al_detener)
tab_monitor = widgets.VBox([
    widgets.HBox([dd_paciente, sl_velocidad]), chk_spo2, widgets.HBox([btn_iniciar, btn_detener]),
    html_mon_estado, img_mon, widgets.HTML('<b>Registro de alertas</b>'), html_mon_log])

# ---------- Pestaña: Evaluación manual ----------
estilo = {'description_width': '110px'}
lay = widgets.Layout(width='460px')
sl_rr = widgets.FloatSlider(value=16, min=4, max=40, step=0.5, description='RR (rpm)', style=estilo, layout=lay)
sl_hr = widgets.FloatSlider(value=80, min=30, max=180, step=1, description='HR (lpm)', style=estilo, layout=lay)
sl_sdnn = widgets.FloatSlider(value=30, min=0, max=200, step=1, description='SDNN (ms)', style=estilo, layout=lay)
sl_rmssd = widgets.FloatSlider(value=25, min=0, max=200, step=1, description='RMSSD (ms)', style=estilo, layout=lay)
chk_man_spo2 = widgets.Checkbox(value=False, description='Incluir SpO2')
sl_spo2 = widgets.FloatSlider(value=97, min=80, max=100, step=0.5, description='SpO2 (%)', style=estilo,
                              layout=lay, disabled=True)
html_manual = widgets.HTML()

def actualizar_manual(*_):
    sl_spo2.disabled = not chk_man_spo2.value
    spo2 = sl_spo2.value if chk_man_spo2.value else None
    r = evaluar_manual(sl_rr.value, sl_hr.value, sl_sdnn.value, sl_rmssd.value, spo2, umbral=sl_umbral.value)
    color = COLORES_NIVEL[r['nivel']]
    pct = r['probabilidad'] * 100
    extra = ''
    if r['banderas']:
        extra += f"<div>⚠️ Bandera roja: {'; '.join(r['banderas'])}</div>"
    if not r['calidad_ok']:
        extra += "<div>⚠️ Con estos valores de HRV el sistema real marcaría la ventana como <b>señal pobre</b>.</div>"
    pts = f"NEWS2 RR: {r['puntos_rr']} pts" + (f" · NEWS2 SpO2: {r['puntos_spo2']} pts" if spo2 is not None else '')
    html_manual.value = (
        f"<div style='font-family:sans-serif;border-radius:10px;padding:14px 18px;background:{color};color:white'>"
        f"<div style='font-size:28px;font-weight:bold'>{r['nivel']}</div>"
        f"<div>Probabilidad de riesgo ({MEJOR_MODELO}): <b>{r['probabilidad']:.3f}</b> · umbral {r['umbral']:.3f}</div>"
        f"<div style='background:rgba(255,255,255,.35);border-radius:6px;height:14px;margin:6px 0'>"
        f"<div style='background:white;height:14px;border-radius:6px;width:{pct:.0f}%'></div></div>"
        f"<div>{pts}</div>{extra}"
        f"<div style='font-size:12px;opacity:.85;margin-top:6px'>Evaluación de una sola ventana: sin histéresis.</div></div>")

for w in (sl_rr, sl_hr, sl_sdnn, sl_rmssd, chk_man_spo2, sl_spo2, sl_umbral):
    w.observe(actualizar_manual, names='value')
actualizar_manual()
tab_manual = widgets.VBox([sl_rr, sl_hr, sl_sdnn, sl_rmssd, widgets.HBox([chk_man_spo2, sl_spo2]), html_manual])

# ---------- Pestaña: Analizar mi señal ----------
up_csv = widgets.FileUpload(accept='.csv', multiple=False, description='Subir CSV')
txt_columna = widgets.Text(value='PLETH', description='Columna PPG')
int_fs = widgets.IntText(value=FS_SIG, description='fs (Hz)')
txt_col_spo2 = widgets.Text(value='', placeholder='opcional', description='Columna SpO2')
btn_analizar = widgets.Button(description='Analizar', button_style='primary', icon='search')
html_arch_estado = widgets.HTML()
img_arch = widgets.Image(format='png', layout=widgets.Layout(width='100%', max_width='820px'))
html_arch_log = widgets.HTML()

def al_analizar(_):
    try:
        if not up_csv.value:
            html_arch_estado.value = "<span style='color:#c62828'>Primero sube un archivo CSV.</span>"; return
        nombre, contenido = leer_archivo_subido(up_csv.value)
        df_sub = pd.read_csv(io.BytesIO(contenido))
        df_sub.columns = [str(c).strip() for c in df_sub.columns]
        col = txt_columna.value.strip()
        if col not in df_sub.columns:
            numericas = df_sub.select_dtypes('number').columns
            col = numericas[-1] if len(numericas) else None
        if col is None:
            html_arch_estado.value = "<span style='color:#c62828'>No encontré una columna numérica de PPG.</span>"; return
        fs = int(int_fs.value)
        ppg = df_sub[col].astype(float).interpolate().bfill().values
        if len(ppg) < VENTANA_S * fs:
            html_arch_estado.value = (f"<span style='color:#c62828'>La señal dura {len(ppg)/fs:.0f} s; "
                                      f"se necesitan al menos {VENTANA_S} s.</span>"); return
        spo2 = None
        cs = txt_col_spo2.value.strip()
        if cs and cs in df_sub.columns:
            tam = VENTANA_S * fs
            spo2 = [df_sub[cs].iloc[w*tam:(w+1)*tam].mean() for w in range(len(ppg) // tam)]
        p, eventos = parametros_actuales(), []
        html_arch_estado.value = f"Analizando <b>{nombre}</b> (columna <b>{col}</b>, {len(ppg)/fs:.0f} s)…"
        def notificar(e):
            eventos.append(e); enviar_webhook(e, txt_webhook.value.strip())
        sistema = ejecutar_sobre_senal(ppg, fs, nombre, spo2_por_ventana=spo2, notificar=notificar, **p)
        hist = sistema.tabla()
        resumen = hist['nivel'].value_counts().reindex(['Normal', 'Vigilancia', 'ALERTA', 'CRÍTICO']).fillna(0).astype(int)
        html_arch_estado.value = (
            html_estado(sistema.historial[-1], nombre, p['umbral']) +
            f"<div style='font-family:sans-serif;margin-top:6px'>Ventanas analizadas: {len(hist)} · "
            + " · ".join(f"{k}: {v}" for k, v in resumen.items())
            + f" · señal pobre: {(hist['calidad'] != 'OK').sum()}</div>")
        img_arch.value = figura_png(hist, p['umbral'], f'{nombre} — {MEJOR_MODELO}')
        html_arch_log.value = html_log(eventos)
    except Exception as ex:
        html_arch_estado.value = f"<span style='color:#c62828'>Error: {ex}</span>"

btn_analizar.on_click(al_analizar)
tab_archivo = widgets.VBox([
    widgets.HTML("CSV con una columna de PPG crudo (y opcionalmente una de SpO2). Mínimo 30 s de señal."),
    widgets.HBox([up_csv, btn_analizar]), widgets.HBox([txt_columna, int_fs]), txt_col_spo2,
    html_arch_estado, img_arch, widgets.HTML('<b>Registro de alertas</b>'), html_arch_log])

# ---------- Panel completo ----------
encabezado = widgets.HTML(
    f"<div style='font-family:sans-serif'><h2 style='margin:4px 0'>🫁 Sistema de alerta respiratoria (PPG + {MEJOR_MODELO})</h2>"
    f"<span style='color:#555'>Modelo: <b>{MEJOR_MODELO}</b> · AUC por paciente: <b>{AUC_MEJOR:.3f}</b> · "
    f"umbral de Youden: <b>{UMBRAL_FINAL:.3f}</b> · riesgo: RR &lt; {RR_MIN} o &gt; {RR_MAX} rpm"
    f"{f', SpO2 &lt; {SPO2_MIN} %' if INCLUIR_SPO2_EN_ETIQUETA else ''}</span></div>")
pestanas = widgets.Tab(children=[tab_monitor, tab_manual, tab_archivo, tab_config])
for i, titulo in enumerate(['Monitor en vivo', 'Evaluación manual', 'Analizar mi señal', 'Configuración']):
    pestanas.set_title(i, titulo)
display(widgets.VBox([encabezado, pestanas]))